### 00. import dependencies

In [18]:
import os
import os.path
import warnings
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from datetime import datetime


# langchain
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, AnyMessage, SystemMessage
from langchain.agents import create_agent
from langchain_core.tools import tool

# langgraph
from langgraph.graph import StateGraph, START, END,add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

# DB + ORM
from sqlalchemy import create_engine,Column,Integer,String
from sqlalchemy.orm import sessionmaker,declarative_base

# vector DB
import chromadb

# GMAIL API
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from email.message import EmailMessage
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
import base64

warnings.filterwarnings("ignore")


### 01. Creating the Environment variables

In [19]:
#### Load the api_key

load_dotenv(dotenv_path='../../.env')

API_KEY = os.getenv("GOOGLE_API_KEY")
DB_URL = os.getenv("DB_URI")
DEFAULT_GEMINI_MODEL = "gemini-3-flash-preview"

if not API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")

if not DB_URL:
    raise ValueError("DB_URL not found in environment variables")

In [20]:
#### initialize the models

llm = ChatGoogleGenerativeAI(model=DEFAULT_GEMINI_MODEL, temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="model/embedding-001")

### 02. RAG pipline

In [21]:
### initialize ChromaDB

chroma_client = chromadb.Client()
collection_name = "enterprise_hr_collection"

try:
    chroma_client.delete_collection(collection_name)
except Exception as e:
    print(f"There no collection to delete : {e}")
collection = chroma_client.create_collection(collection_name)

In [22]:
### Load the documents

file_paths = [
    "../data/raw/hr_policy.pdf",
    "../data/raw/complaints_policy.pdf",
    "../data/raw/leave_policy.pdf"
]

documents = []
for path in file_paths:
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        documents.extend(loader.load())
    else:
        print(f"Warning: {path} not found. Skipping.")

In [23]:
### Chunking and embedding

if documents:
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ".", " "],
        chunk_size=400,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False,
    )
    chunks =text_splitter.split_documents(documents)
    texts = [c.page_content for c in chunks]
    metadatas = [c.metadata for c in chunks]
    ids = [f"doc_{i}" for i in range(len(chunks))]

    collection.add(documents=texts, metadatas=metadatas, ids=ids)
    print(f"Added {len(texts)} chunks into chromaDB.")
else:
    print("No documents to add.")

Added 17 chunks into chromaDB.


### 03. database setup (sample)

In [24]:
Base =  declarative_base()

class EmployeeModel(Base):
    __tablename__ = "employees"
    employee_id = Column(String, primary_key=True)
    name = Column(String)
    pto_days = Column(Integer)
    sick_days = Column(Integer)
    email = Column(String)

engine = create_engine(DB_URL)
SessionLocal= sessionmaker(bind=engine)

Base.metadata.create_all(bind=engine)

### 04. Creating tools

In [25]:
### connect to the gmail api with credentials

SCOPES = ['https://www.googleapis.com/auth/gmail.send']

def get_gmail_service():
    """Gets valid user credentials from storage or prompts the user."""
    creds = None
    ### check the token is available in local machine or not
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)

    ### if creds is not have or not valid then create a token and save to token.json
    if not creds or not creds.valid:

        if creds and creds.expired and creds.refresh_token:
            ## Try refreshing login
            creds.refresh(Request())
        else:
            ## if refresh not possible, ask user again to log in
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    return build('gmail', 'v1', credentials=creds)



In [26]:
@tool
def check_sql_leave_balance(employee_id:str)-> str:
    """Check PTO and sick leave balance, name, and email for a given employee ID."""
    db = SessionLocal()
    try:
        emp = db.query(EmployeeModel).filter_by(employee_id = employee_id).first()
        if not emp:
            return f"Employee {employee_id} not found."

        # UPDATE THIS LINE to include email and ID
        return f"Name: {emp.name}, Email: {emp.email}, ID: {employee_id}. Balance: {emp.pto_days} PTO, {emp.sick_days} sick leaves."
    finally:
        db.close()


@tool
def hr_vector_search(query:str)->str:
    """Search HR policy documents using semantic vector search."""
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    if results['documents'] and results['documents'][0]:
        docs = results["documents"][0]
        metas = results["metadatas"][0]

        formatted_context = ""

        for i in range(len(docs)):
            source_path = metas[i].get('source', 'Unknown Document')
            filename = os.path.basename(source_path)
            page_num = metas[i].get('page', -1) + 1

            formatted_context += f"--- SOURCE: {filename} (Page {page_num}) ---\n{docs[i]}\n\n"
        return formatted_context

    return "No relevant policy found."

@tool
def draft_and_send_email(recipient: str, subject: str, body: str, is_confirmed: bool = False) -> str:
    """Draft an email and send it only after user confirmation."""

    # 1. The Draft Phase
    if not is_confirmed:
        return f"""
EMAIL_DRAFT:
To: {recipient}
Subject: {subject}

{body}

Reply YES to confirm sending.
"""

    # 2. The Sending Phase (executes only when is_confirmed=True)
    try:
        # Get the authenticated Gmail service
        service = get_gmail_service()

        # Construct the MIME email
        message = EmailMessage()
        message.set_content(body)
        message['To'] = recipient
        message['Subject'] = subject

        # Encode the message securely
        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
        create_message = {'raw': encoded_message}

        # Send it!
        service.users().messages().send(userId="me", body=create_message).execute()

        print(f"\n[ACTUAL EMAIL SENT] → Successfully sent to {recipient}")
        return "SUCCESS: Email sent via your personal Gmail account."

    except Exception as e:
        print(f"\n[EMAIL FAILED] → {e}")
        return f"FAILED to send email: {str(e)}"

### 05. Create router layer 1

In [27]:
### making state
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    user_context: dict

In [28]:
# intent_profiles = {
#     "leave_agent" : ["leave","vacation","pto","holiday"],
#     "complaint_agent" : ["complaint","issue","harassment"],
#     "hr_agent":["policy","salary","benefits","dress code"]
# }
intent_profiles = {
    "leave_agent" : ["leave", "vacation", "pto", "holiday", "sick"],
    "complaint_agent" : ["complaint", "complain", "issue", "harassment", "report"],
    "hr_agent": ["policy", "salary", "benefits", "dress code"]
}

def semantic_router(state: State):
    messages = state.get("messages", [])

    # extract text from LangChain message content
    def safe_extract_text(content):
        if isinstance(content, list):
            return " ".join([item.get("text", "") for item in content if isinstance(item, dict)]).lower()
        return str(content).lower()

    # get the current user text safely
    user_text = ""
    for msg in reversed(messages):
        if getattr(msg, 'type', '') == 'human':
            user_text = safe_extract_text(msg.content).strip()
            break

    if not user_text:
        return Command(goto="hr_agent")

    ### check for active conversation flow (seeing the last ai message)
    history_to_check = messages[:-1] if len(messages) > 1 else []
    last_ai_msg = ""
    for msg in reversed(history_to_check):
        if getattr(msg, 'type', '') == 'ai':
            last_ai_msg = safe_extract_text(msg.content)
            break # Found the most recent AI response

    # If the AI was just talking about drafting an email, stay with the COMM_AGENT
    # if "reason for" in last_ai_msg or "draft" in last_ai_msg or "reply yes" in last_ai_msg:
    #     print("[Router] Active email drafting flow detected! Routing directly to -> COMM_AGENT")
    #     return Command(goto="comm_agent")
    if any(phrase in last_ai_msg for phrase in ["reason for", "draft", "reply yes", "incident", "complaining about"]):

        # Decide if we are in a leave flow or a complaint flow based on the previous agent's context
        # A simple heuristic: if the AI was asking about dates/incidents for a complaint, route to complaint
        if "incident" in last_ai_msg or "complaining about" in last_ai_msg:
             print("[Router] Active complaint drafting flow detected! Routing directly to -> COMPLAINT_AGENT")
             return Command(goto="complaint_agent")
        else:
             print("[Router] Active email drafting flow detected! Routing directly to -> COMM_AGENT")
             return Command(goto="comm_agent")



    # 3. Robust Keyword Routing
    print(f"[Router] Evaluating user input: '{user_text}'")
    for agent, keywords in intent_profiles.items():
        if any(keyword in user_text for keyword in keywords):
            print(f"[Router] Intent matched! Routing to -> {agent.upper()}")
            return Command(goto=agent)

    # return to the HR_Agent
    print("[Router] No keywords matched. Defaulting to -> HR_AGENT")
    return Command(goto="hr_agent")

### 06. making Agent

In [29]:
LEAVE_PROMPT = """
You are the Enterprise Leave Agent.
Check the message history for a System Note containing the user's Employee ID.
Use that specific ID when calling the check_sql_leave_balance tool. Do NOT ask the user for their ID.

Answer leave queries using your tools.
If the user asks you to email, notify, or message someone → DO NOT draft it yourself.
Instead, output exactly this phrase: HANDOFF_TO_COMMUNICATION
"""
COMM_PROMPT = """
You are the Enterprise Communication Agent.
Your job is to draft and send emails using the draft_and_send_email tool.

CRITICAL INSTRUCTIONS:
1. RECIPIENT: Always send the email to the hardcoded address: 'mhdatheeq0@gmail.com'. Do not ask the user for the recipient.
2. BALANCE CHECK (MANDATORY): You MUST use the check_sql_leave_balance tool to fetch the employee's Name, Email, and current Leave Balances.
3. CONDITIONAL INFORMATION GATHERING: Before drafting, you MUST know 3 things: Exact Dates, Total Number of Days, and Reason.
   - STEP A (Extract): Analyze the user's request. Did they already state the number of days or exact dates (e.g., "tomorrow", "for 3 days")? Did they give a reason?
   - STEP B (Skip): If they ALREADY provided the days/dates and reason, do NOT ask them again. Calculate the calendar dates using the System Note and proceed to Step 4.
   - STEP C (Ask): If any of those 3 things are missing, politely ASK the user ONLY for the missing information. DO NOT call your email tool until you have all three.
4. VALIDATION GATE: You must mathematically compare the total requested days against the employee's available PTO or Sick leave balance.
   - IF REQUEST > BALANCE: DO NOT draft the email. Politely decline and state their current balance.
   - IF REQUEST <= BALANCE: Proceed to Step 5.
5. SIGNATURE FORMAT: You MUST sign off the email using exactly this format:

   Sincerely,
   [Employee Name]
   Employee ID: [Employee ID]
   Email: [Employee Email]

6. EXECUTION: Once validated and all details are gathered, call draft_and_send_email with is_confirmed=False.
7. CONFIRMATION LOOP: If the user replies "YES", call the tool again with is_confirmed=True.
"""
COMPLAINT_PROMPT = """
You are the Enterprise Grievance & Complaint Agent.
Your job is to securely log official workplace complaints by sending an email to HR using the draft_and_send_email tool.

CRITICAL INSTRUCTIONS:
1. RECIPIENT: Always send the complaint email to the hardcoded HR address: 'mhdatheeq0@gmail.com'. Do not ask the user for the recipient.
2. GATHER DETAILS (MANDATORY): Before drafting the complaint, you MUST ask the user for:
   - The name of the person they are complaining about.
   - The date(s) the incident(s) occurred.
   - A detailed description of the incident.
3. CONDITIONAL GATHERING:
   - Analyze the user's initial request. If they are missing any of the 3 items above, politely ASK for the missing information.
   - DO NOT draft the email until you have all the details.
4. SENDER INFO: Use the check_sql_leave_balance tool to fetch the complaining employee's actual Name and Email address to use in the signature.
5. SUBJECT LINE: Write a professional subject line indicating this is a formal grievance.
6. EXECUTION: Once all details are gathered, call draft_and_send_email with is_confirmed=False.
7. CONFIRMATION LOOP: If the user replies "YES" to confirm the draft, you MUST call the tool again with is_confirmed=True to officially file the complaint.
"""
leave_agent = create_agent(
    model=llm,
    tools=[check_sql_leave_balance,hr_vector_search],
    system_prompt=LEAVE_PROMPT)

complaint_agent = create_agent(
    model=llm,
    tools=[hr_vector_search,draft_and_send_email,check_sql_leave_balance],
    system_prompt=COMPLAINT_PROMPT
)
hr_agent = create_agent(
    model=llm,
    tools=[hr_vector_search]
)

comm_agent = create_agent(
    model=llm,
    tools=[draft_and_send_email, check_sql_leave_balance],
    system_prompt=COMM_PROMPT
)

### 07. making node logics

In [30]:
# def run_leave_agent(state: State):
#     result = leave_agent.invoke(state)
#     messages = result["messages"]
#     last_msg_content = messages[-1].content
#
#     if "HANDOFF_TO_COMMUNICATION" in last_msg_content:
#         print("[Handoff] Leave Agent requesting Communication Agent...")
#
#         ## bridging message to trigger the next agent into action
#         bridge_message = HumanMessage(
#             content="System Note: Handoff received. Please review the user's request above and use your tools to draft the email or notification."
#         )
#
#         ## Append it to the history before sending it to the comm_agent
#         messages.append(bridge_message)
#
#         return Command(update={"messages": messages}, goto="comm_agent")
#
#     return Command(update={"messages": messages}, goto=END)
def run_leave_agent(state: State):
    result = leave_agent.invoke(state)
    messages = result["messages"]

    # Safely extract the text, handling both strings and lists
    raw_content = messages[-1].content
    if isinstance(raw_content, list):
        content_str = " ".join([item.get("text", "") for item in raw_content if isinstance(item, dict)])
    else:
        content_str = str(raw_content)

    # Now check the safely extracted string
    if "HANDOFF_TO_COMMUNICATION" in content_str:
        print("[Handoff] Leave Agent requesting Communication Agent...")

        ## bridging message to trigger the next agent into action
        bridge_message = HumanMessage(
            content="System Note: Handoff received. Please review the user's request above and use your tools to draft the email or notification."
        )

        ## Append it to the history before sending it to the comm_agent
        messages.append(bridge_message)

        return Command(update={"messages": messages}, goto="comm_agent")

    return Command(update={"messages": messages}, goto=END)
def run_hr_agent(state:State):
    result = hr_agent.invoke(state)
    return Command(update= {"messages": result["messages"]},goto=END)

def run_complaint_agent(state:State):
    result = complaint_agent.invoke(state)
    return Command(update= {"messages": result["messages"]},goto=END)

def run_comm_agent(state:State):
    result = comm_agent.invoke(state)
    return Command(update= {"messages": result["messages"]},goto=END)

### 08. context injection (add the employee id )

In [31]:
# def inject_context(state:State):
#     state["user_context"] = {"employee_id": "EMP001"}
#     return state
def inject_context(state: State):
    messages = state.get("messages", [])

    # Get today's actual date with the day of the week
    today_date = datetime.now().strftime("%A, %B %d, %Y")

    if len(messages) == 1:
        system_note = SystemMessage(
            content=f"System Note: The user's Employee ID is EMP001. Today's exact date is {today_date}. You are highly capable of calculating tricky relative dates (e.g., 'tomorrow', 'next week same day', 'in 3 days', 'next Monday') based on today's date. Always convert relative terms to exact calendar dates."
        )

        return {
            "user_context": {"employee_id": "EMP001"},
            "messages": [system_note]
        }

    return {"user_context": {"employee_id": "EMP001"}}

In [32]:
builder = StateGraph(State)

builder.add_node("context", inject_context)
builder.add_node("router", semantic_router)
builder.add_node("leave_agent", run_leave_agent)
builder.add_node("complaint_agent", run_complaint_agent)
builder.add_node("hr_agent", run_hr_agent)
builder.add_node("comm_agent", run_comm_agent)

builder.add_edge(START,"context")
builder.add_edge("context","router")

app = builder.compile(checkpointer=MemorySaver())

In [33]:
config = {"configurable": {"thread_id": "user_1"}}


def chat(user_input):
    events = app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="values"
    )

    final = None
    for e in events:
        final = e["messages"][-1]

    output = final.content


    if isinstance(output, list):
        clean_text = " ".join([item["text"] for item in output if item.get("type") == "text"])
    else:
        clean_text = output

    print(f"\n🤖: {clean_text}")


### 09. testing

In [34]:
chat("i want make a complain for one employee.")

[Router] Evaluating user input: 'i want make a complain for one employee.'
[Router] Intent matched! Routing to -> COMPLAINT_AGENT

🤖: I can certainly help you with that. To ensure your grievance is handled professionally and thoroughly, I need a few more details to include in the formal report:

1.  **Who** is the person you are complaining about?
2.  **When** did the incident(s) occur? (Please provide specific dates).
3.  **What** happened? (Please provide a detailed description of the incident).

Once you provide these details, I will draft the formal complaint for your review.


In [35]:
chat("i want report about the emp002 Nuvan")

[Router] Active complaint drafting flow detected! Routing directly to -> COMPLAINT_AGENT

🤖: I have noted that the complaint is regarding **Nuvan (EMP002)**. 

To proceed, I still need the following information:
1.  **When** did the incident(s) occur?
2.  **What** happened? (Please provide a detailed description of the incident). 

Once I have these details, I can draft the formal grievance for you.


In [36]:
chat("it happend in last friday, he is leave early in office at 3.00.")

[Router] Active complaint drafting flow detected! Routing directly to -> COMPLAINT_AGENT

🤖: I have drafted the formal grievance for you. Here are the details:

**To:** mhdatheeq0@gmail.com  
**Subject:** Formal Grievance - Incident Report regarding Nuvan (EMP002)  

**Body:**  
Dear HR,  

I am writing to formally log a grievance regarding an incident involving Nuvan (EMP002).  

**Incident Details:**  
- **Date:** March 20, 2026  
- **Description:** The employee left the office early at 3:00 PM.  

This report is submitted by:  
**Name:** kasun  
**Email:** kasun@gmail.com  

Please let me know the next steps in this process.  

Sincerely,  
kasun  

**Please reply "YES" to confirm and officially file this complaint.**


In [37]:
chat("yes")

[Router] Active complaint drafting flow detected! Routing directly to -> COMPLAINT_AGENT

[ACTUAL EMAIL SENT] → Successfully sent to mhdatheeq0@gmail.com

🤖: The formal grievance has been successfully sent to HR. Is there anything else I can assist you with?


In [17]:
chat("How many leave days do I have?")

[Router] Evaluating user input: 'how many leave days do i have?'
[Router] Intent matched! Routing to -> LEAVE_AGENT

🤖: You currently have 4 PTO days and 1 sick leave day remaining.


In [19]:
chat("Send an email requesting leave nextweek sameday ")

[Router] Evaluating user input: 'send an email requesting leave nextweek sameday'
[Router] Intent matched! Routing to -> LEAVE_AGENT
[Handoff] Leave Agent requesting Communication Agent...

🤖: To proceed with your leave request for next week, I need a few more details:

1.  **Total Number of Days:** How many days are you requesting?
2.  **Reason:** What is the reason for your leave?

Once I have this information, I can verify your balance and draft the email for you. (Note: "Next week same day" refers to Thursday, April 2, 2026.)


In [20]:
chat("i am going a family trip")

[Router] Active email drafting flow detected! Routing directly to -> COMM_AGENT

🤖: I have the reason (family trip) and the date (Thursday, April 2, 2026), but I still need to know the **total number of days** you are requesting for this leave. 

Could you please confirm how many days you need?


In [21]:
chat("yes")

[Router] Evaluating user input: 'yes'
[Router] No keywords matched. Defaulting to -> HR_AGENT

🤖: I apologize for the confusion. Since you mentioned "next week same day," I am assuming you are requesting **one day** off for Thursday, April 2, 2026.

Here is a draft for your leave request email:

***

**Subject:** Leave Request - [Your Name] - April 2, 2026

Dear [Manager's Name],

I am writing to request a day of PTO for next week, Thursday, April 2, 2026, for a family trip.

I have ensured that my current tasks are up to date and will make sure any urgent matters are covered before my absence. Please let me know if this request is approved or if you need any further information.

Thank you for your consideration.

Best regards,

Kasun

***

**Would you like me to send this email for you, or would you like to make any changes?**


In [22]:
chat("yes")

[Router] Active email drafting flow detected! Routing directly to -> COMM_AGENT

[ACTUAL EMAIL SENT] → Successfully sent to mhdatheeq0@gmail.com

🤖: The email requesting your leave for Thursday, April 2, 2026, has been sent successfully.
